# Gaussian Mixture Models (GMMs)

A Gaussian Mixture Model is a flexible, probabilistic clustering and density‑estimation tool that represents data as coming from a combination of several Gaussian distributions. Unlike hard‑clustering methods, GMMs assign each point a soft (probabilistic) membership across clusters, allowing for overlapping, elliptical cluster shapes.

<br>

<p align="center">
<img src="visualizations/GMM.png" width="600">
</p>

* **Goal:** Model the overall data density $p(\mathbf{x})$ as a weighted sum of $K$ Gaussians

  $$
    p(\mathbf{x}) = \sum_{k=1}^K \pi_k \,\mathcal{N}(\mathbf{x}\mid\mu_k,\Sigma_k),
  $$

  where $\{\pi_k\}$ are mixing weights, $\{\mu_k\}$ means, and $\{\Sigma_k\}$ covariances.
* **Soft Clustering:** Each data point $\mathbf{x}_n$ carries a “responsibility” $\gamma_{nk}\in[0,1]$ for each component $k$.


### The Generative Story

1. **Select a component** $k$ with probability $\pi_k$.
2. **Draw a sample** $\mathbf{x}\sim\mathcal{N}(\mu_k,\Sigma_k)$.

This models how the data could have been generated by first choosing one of $K$ “latent” processes, then sampling from that process’s Gaussian.

### Why “Mixture”?

* **Multiple sources:** Real‑world data often arise from heterogeneous processes.
* **Weighted sums:** We mix $K$ Gaussians so that the overall shape can approximate complex, multimodal densities.

---

## Optimization: Expectation–Maximization (EM)

Because directly maximizing
$\sum_n\ln\bigl[\sum_k\pi_k\,\mathcal{N}(\mathbf{x}_n)\bigr]$
is intractable, GMMs use EM:

1. **Initialization:**
   Choose initial guesses for parameters $\{\pi_k^{(0)}, \mu_k^{(0)}, \Sigma_k^{(0)}\}$.

2. **E‑step:** (Expectation)


   $$
     \gamma_{nk}
     = \frac{\pi_k\,\mathcal{N}(\mathbf{x}_n\mid\mu_k,\Sigma_k)}
            {\sum_j\pi_j\,\mathcal{N}(\mathbf{x}_n\mid\mu_j,\Sigma_j)}.
   $$
3. **M‑step:** (Maximization)

   $$
     N_k = \sum_n \gamma_{nk},\quad
     \pi_k \leftarrow \tfrac{N_k}{N},\quad
     \mu_k \leftarrow \tfrac1{N_k}\sum_n \gamma_{nk}\,\mathbf{x}_n,\quad
     \Sigma_k \leftarrow \frac{1}{N_k}\sum_{n=1}^N \gamma_{nk}\,(\mathbf{x}_n - \mu_k)(\mathbf{x}_n - \mu_k)^\top
   $$
4. **Convergence check:**  Repeat E and M until the log‑likelihood improvement falls below a threshold.

---

## Covariance Structures

Choosing how “rich” each $\Sigma_k$ can be affects flexibility and parameter count:

* **Full:** each cluster its own arbitrary $\Sigma_k$.
* **Tied:** all clusters share one covariance.
* **Diagonal:** $\Sigma_k$ diagonal (axis‑aligned ellipsoids).
* **Spherical:** $\Sigma_k = \sigma_k^2 I$ (isotropic balls).

---

## Model Selection

Selecting *K* is crucial. Common approaches:

* **Bayesian Information Criterion (BIC)** or **Akaike Information Criterion (AIC)**
  Evaluate the penalized log‑likelihood
  $\text{BIC} = -2\,\mathcal{L} + p\,\ln N,$
  where *p* = number of free parameters. Choose *K* minimizing BIC.
* **Cross‑validation**
  Estimate held‑out log‑likelihood for different *K*.


---

## Recovering K‑Means

Under the special limit:

1. **Isotropic covariances:** $\Sigma_k = \sigma^2 I$.
2. **Equal weights:** $\pi_k = 1/K$.
3. **Variance → 0:** $\sigma^2\to0$.

Then responsibilities $\gamma_{nk}$ become hard 0/1 assignments to the nearest mean, and the EM updates reduce exactly to Lloyd’s K‑Means iterations.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import multivariate_normal


class GMM:
    def __init__(self, n_components, n_iter=100, tol=1e-4):
        self.K = n_components
        self.n_iter = n_iter
        self.tol = tol

    def initialize_parameters(self, X):
        N, D = X.shape
        self.N, self.D = N, D
        self.pi = np.ones(self.K) / self.K
        indices = np.random.choice(N, self.K, replace=False)
        self.mu = X[indices]
        self.sigma = np.array([np.eye(D) for _ in range(self.K)])

    def e_step(self, X):
        self.gamma = np.zeros((self.N, self.K))
        for k in range(self.K):
            rv = multivariate_normal(self.mu[k], self.sigma[k])
            self.gamma[:, k] = self.pi[k] * rv.pdf(X)
        self.gamma /= self.gamma.sum(axis=1, keepdims=True)

    def m_step(self, X):
        N_k = self.gamma.sum(axis=0)
        self.pi = N_k / self.N
        self.mu = (self.gamma.T @ X) / N_k[:, np.newaxis]
        for k in range(self.K):
            diff = X - self.mu[k]
            self.sigma[k] = (self.gamma[:, k][:, np.newaxis] * diff).T @ diff / N_k[k]
            self.sigma[k] += np.eye(self.D) * 1e-6  # regularization

    def compute_log_likelihood(self, X):
        ll = 0
        for k in range(self.K):
            rv = multivariate_normal(self.mu[k], self.sigma[k])
            ll += self.pi[k] * rv.pdf(X)
        return np.sum(np.log(ll))

    def fit(self, X):
        self.initialize_parameters(X)
        self.log_likelihoods = []
        for i in range(self.n_iter):
            self.e_step(X)
            self.m_step(X)
            ll = self.compute_log_likelihood(X)
            self.log_likelihoods.append(ll)
            if (
                i > 0
                and np.abs(self.log_likelihoods[-1] - self.log_likelihoods[-2])
                < self.tol
            ):
                break

    def predict(self, X):
        probs = np.zeros((X.shape[0], self.K))
        for k in range(self.K):
            rv = multivariate_normal(self.mu[k], self.sigma[k])
            probs[:, k] = self.pi[k] * rv.pdf(X)
        return np.argmax(probs, axis=1)